# 3. Ingeniería de Datos con Pandas
----

### Parte 3.1 
----
Cargar el .csv usando pandas.read_csv, luego especificar el encoding y parse_dates. Mostrar los datos usando df.info y comentar algunas observaciones.

In [ ]:
import pandas as pd
import csv

ventas_techventas = "../data/ventas_techventas.csv"
encoding = "latin-1"

dataFrame_raw = pd.read_csv(
    ventas_techventas, 
    sep=",",
    parse_dates=["fecha"], 
    encoding=encoding, 
    quoting=csv.QUOTE_NONE
)

dataFrame = dataFrame_raw.copy()

dataFrame["order_id"] = dataFrame["order_id"].str.lstrip('"')
dataFrame["order_id"] = pd.to_numeric(dataFrame["order_id"])

dataFrame["vendedor"] = dataFrame["vendedor"].str.replace("Ana Lï¿½ï¿", "Ana López")
dataFrame['vendedor'] = dataFrame['vendedor'].str.strip('"')

dataFrame['producto'] = dataFrame['producto'].str.replace('""', '"')

dataFrame.info()

#### Observaciones sobre la carga del dataset
En la celda anterior se realiza la carga del csv, con el encoding `latin-1` dado que este es el encoding que le permite a pandas leer el dataset, se hace un parse_dates de la columna fecha y además se agrega el parámetro `quoting=csv.QUOTE_NONE` debido a que el csv contiene 11 filas que se encuentran entre comillas, y al cargar el dataset sin este parámetro, toma toda la fila como un string de la columna order_id, el resto de columnas quedan nulas, por lo que se le indica a pandas que por el momento ignore las comillas.

Una vez se creó el data frame, se eliminan las comillas residuales de la columna order_id y se convierte a int. 
Al pasar el contenido del dataset por herramientas como la libreria chardet para confirmar su codificación, aparecen un par de errores de codificación, especialmente en el nombre de Ana López, sin embargo la librería indica que sigue codificado en latin-1, por lo que decidí reemplazar el error en el nombre por el nombre correcto que se indica en el documento para evitar problemas con el ejercicio.
Se elimina la comilla extra presente en algunos de los productos y las comillas residuales de la columna vendedor.

### Parte 3.2
----
Detectar nulos con isna(). Eliminar/imputar justificando. Documentar cuántos registros se descartaron y por qué.

In [ ]:
productos_nulos = dataFrame["producto"].isna().sum()
cantidades_negativas = (dataFrame["cantidad"] <= 0).sum()

print(f"Productos nulos en el dataset: {productos_nulos}")
print(f"Pedidos con cantidades negativas: {cantidades_negativas}")

Se van a imputar 2 registros.
Continuando con el proceso de limpieza que se llevaba realizando desde la parte 1 del ejercicio, se van a imputar los valores. En primer lugar, considerando que el dato nulo se trata de un nombre de producto, y que el dataset no cuenta con una tabla de productos donde sea posible encontrar una coincidencia de precio, categoría y descuento, la mejor opción añadir una etiqueta por defecto que se pueda filtrar en el futuro para realizar la corrección y evitar eliminar el registro.
Por otro lado, respecto al pedido que tiene cantidad negativa, nuevamente, es mejor aplicarle una transformación de valor absoluto, no se tiene certeza de que el valor una vez modificado sea el valor que se esperaba, sin embargo, es la opción que permite conservar la mayor cantidad de información valiosa para la empresa, en lugar de eliminar el registro y que dicha información se pierda o se omita.

In [ ]:
dataFrame["producto"] = dataFrame["producto"].fillna("Producto Desconocido")
dataFrame["cantidad"] = dataFrame["cantidad"].abs()
dataFrame.info()

### Parte 3.3
----
Crear las columnas de ingreso bruto, ingreso neto y mes. Agrupar por mes con groupby + agg

In [ ]:
dataFrame["ingreso_bruto"] = dataFrame["cantidad"] * dataFrame["precio_unitario"]
dataFrame["ingreso_neto"] = dataFrame["ingreso_bruto"] * (1 - dataFrame["descuento"])
dataFrame["mes"] = dataFrame["fecha"].dt.to_period('M')

dataFrame.head()

In [ ]:
resumen_mensual = dataFrame.groupby("mes").agg(
    ingreso_total=("ingreso_neto", "sum"), 
    numero_pedidos=("order_id", "count")
).reset_index()
display(resumen_mensual)

### Parte 3.4
----
Usar concat o merge para unir el resumen mensual con una tabla de metas. Calcular el porcentaje de cumplimiento

**Nota: la meta mensual de la empresa la obtuve sumando las metas mensuales de todos los vendedores que aparecían en la tabla de la parte 2: SQL discovery.**

In [ ]:
# Metas de Ana + Carlos + Luis + Maria
meta_mensual_combinada = 8000 + 7500 + 8000 + 7000

metas_techventas = pd.DataFrame({
    'mes' : pd.period_range(start='2024-01', end='2024-06', freq='M'),
    'meta_mensual' : [meta_mensual_combinada] * 6
})

resumen_final = pd.merge(resumen_mensual, metas_techventas, on="mes", how="inner")

resumen_final['porcentaje_cumplimiento'] = round(
    (resumen_final['ingreso_total'] / resumen_final['meta_mensual']) * 100, 2
)

resumen_final

### Parte 3.5
----
Exportar el dataFrame con to_sql() y verifica de vuelta con un select count(*) from "ventas".

**Nota: en el ejercicio no se pide específicamente, pero también voy a exportar el dataFrame de resumen final del punto anterior para tener los datos juntos**

In [ ]:
import os
import sqlite3

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
output_db = os.path.join(base_dir, "output", "techventas.db")

connection = sqlite3.connect(output_db)

try:
    dataFrame['mes'] = dataFrame['mes'].astype(str)
    resumen_final['mes'] = resumen_final['mes'].astype(str)

    dataFrame.to_sql("ventas", connection, if_exists='replace', index=False)
    resumen_final.to_sql("resumen_mensual", connection, if_exists="replace", index=False)

    dataFrame_ventas = pd.read_sql_query('SELECT COUNT(*) AS numero_pedidos FROM ventas', connection)
    dataFrame_resumen_mensual = pd.read_sql_query('SELECT * FROM resumen_mensual', connection)
    
    print("Tabla de ventas:")
    display(dataFrame_ventas)

    print("Tabla de resumen mensual:")
    display(dataFrame_resumen_mensual)

finally:
    connection.close()